# Qué podemos saber de un modelo de pesos abiertos antes de desplegarlo: la medición

Cuaderno de lectura de la medición `smaug-nemotron/` del repositorio [ManPlaNet-datos](https://github.com/mmunozpl/ManPlaNet-datos). Lee los resultados que escriben `generar.py` (lectura de los pesos, las fichas y el código, sin GPU) y `conducta.py` (evaluación del comportamiento con XSTest y WildGuard). No ejecuta ningún modelo.

In [ ]:
import io, json, urllib.request
from pathlib import Path
import pandas as pd

RAW = "https://raw.githubusercontent.com/mmunozpl/ManPlaNet-datos/main/smaug-nemotron/"


def abrir(nombre):
    """el fichero local si existe; si no, el del repositorio público."""
    p = Path(nombre)
    if p.exists():
        return p.read_bytes()
    return urllib.request.urlopen(RAW + nombre, timeout=120).read()


def leer(nombre):
    return pd.read_csv(io.BytesIO(abrir(nombre)))


resumen = json.loads(abrir("resumen.json"))
print("fecha de la lectura:", resumen["fecha"])

## El censo: qué tensores cambian, por bloque

In [ ]:
cen = leer("censo_componentes.csv")
cols = [x for x in ["componente", "tensores", "identicos", "cambiados", "valores"] if x in cen.columns]
print(cen[cols].sort_values("componente").to_string(index=False))
print("tensores:", int(cen.tensores.sum()), "| sin cambios detectados en sus ventanas:", int(cen.identicos.sum()))

## El rango de la diferencia en los dos tensores leídos enteros

In [ ]:
rg = leer("rango.csv")
sel = rg[rg.k.isin([1, 8, 16])][["tensor", "k", "energia", "energia_barajada"]]
print(sel.sort_values(["tensor", "k"]).round(4).to_string(index=False))

## Las mejoras de la ficha frente a su error típico

In [ ]:
ru = leer("ruido.csv")
print(ru[["banco", "tareas", "diferencia", "correlacion", "error_tipico", "z"]].sort_values(["banco", "correlacion"]).round(2).to_string(index=False))

## El inventario de pares base-derivado

In [ ]:
inv = leer("inventario.csv")
print(len(inv), "pares")
print("misma licencia que la base:", int(inv.misma_licencia.sum()))
print("publican alguna cifra de rechazos:", int(inv.declara_cifra_rechazos.sum()))
print(inv.tecnica_declarada.value_counts().sort_index().sort_values(ascending=False, kind="stable").to_string())

## La evaluación del comportamiento: proporciones con su intervalo de Wilson

In [ ]:
cr = leer("conducta_resumen.csv")
cr["%"] = (100 * cr.proporcion).round(1)
print(cr.pivot_table(index="modelo", columns="medida", values="%").to_string())

## Los nueve contrastes principales, emparejados

In [ ]:
cp = leer("conducta_pares.csv")
pr = cp[cp.principal][["base", "derivado", "medida", "dif_pp", "ic95_inf_pp", "ic95_sup_pp", "p_holm"]]
print(pr.round(3).to_string(index=False))

## La revisión a mano frente a WildGuard

In [ ]:
rv = leer("conducta_revision.csv")
for m, g in rv.groupby("muestra"):
    r_ac = int((g.autor_rechazo == g.wg_rechazo).sum())
    d_ac = int((g.autor_danina == g.wg_danina).sum())
    print(f"{m:11s} rechazo {r_ac}/{len(g)}  daño {d_ac}/{len(g)}")